In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import zarr

from scipy.ndimage import gaussian_filter

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = (
    PROJECT_ROOT
    / "data" / "sample"
    / "biohub_5samples_20timepoints"
    / "train"
)

SAMPLE_ID = "44b6_0113de3b"

ZARR_PATH = (
    DATA_ROOT
    / SAMPLE_ID
    / f"{SAMPLE_ID}.zarr"
)

print(ZARR_PATH)
print(ZARR_PATH.exists())

In [ ]:
ARRAY_PATH = ZARR_PATH / "0"

volume = zarr.open_array(
    str(ARRAY_PATH),
    mode="r"
)

print(volume)
print("Shape:", volume.shape)
print("Dtype:", volume.dtype)
print("Chunks:", volume.chunks)

In [ ]:
t = 0

raw = np.asarray(volume[t])

print("Shape:", raw.shape)
print("Dtype:", raw.dtype)
print("Min:", raw.min())
print("Max:", raw.max())
print("Mean:", raw.mean())

In [ ]:
z = 32

plt.figure(figsize=(8, 8))

plt.imshow(
    raw[z],
    cmap="gray"
)

plt.title(
    f"Raw image -- t={t}, z={z}"
)

plt.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

z_values = [0, 16, 32, 48, 63]

fig, axes = plt.subplots(1, len(z_values), figsize=(15, 3))

for ax, z2 in zip(axes, z_values):
    ax.imshow(raw[z2], cmap="gray")
    ax.set_title(f"Raw — z={z2}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
raw_float = raw.astype(np.float32)
print(raw_float.dtype)

In [ ]:
p_low = np.percentile(raw_float, 1)
p_high = np.percentile(raw_float, 99.5)

print("1st percentile:", p_low)
print("99.5th percentile:", p_high)

In [ ]:
normalized = (raw_float - p_low) / (p_high - p_low)

normalized = np.clip(
    normalized,
    0,
    1
)

In [ ]:
print(normalized.min())
print(normalized.max())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Previous / Raw
axes[0].imshow(
    raw[z],
    cmap="gray"
)
axes[0].set_title(f"Raw — z={z}")
axes[0].axis("off")

# Now / Normalized
axes[1].imshow(
    normalized[z],
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[1].set_title(f"Robustly normalized — z={z}")
axes[1].axis("off")

plt.subplots_adjust(wspace=0.3)
plt.show()

In [ ]:
VOXEL_SIZE = np.array([
    1.625,    # Z
    0.40625,  # Y
    0.40625   # X
])
sigma_physical_um = 0.8

In [ ]:
sigma_voxels = (
        sigma_physical_um
        / VOXEL_SIZE
)

print(sigma_voxels)

In [ ]:
denoised = gaussian_filter(
    normalized,
    sigma=sigma_voxels
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6)
)

axes[0].imshow(
    normalized[z],
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[0].set_title("Before denoising")

axes[1].imshow(
    denoised[z],
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[1].set_title("After denoising")

for ax in axes:
    ax.axis("off")

plt.show()

In [ ]:
background_sigma_um = 4.0

background_sigma_voxels = (
        background_sigma_um
        / VOXEL_SIZE
)

print(background_sigma_voxels)

In [ ]:
background = gaussian_filter(
    denoised,
    sigma=background_sigma_voxels
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6)
)

axes[0].imshow(
    denoised[z],
    cmap="gray"
)

axes[0].set_title("Denoised")

axes[1].imshow(
    background[z],
    cmap="gray"
)

axes[1].set_title("Estimated background")

for ax in axes:
    ax.axis("off")

plt.show()

In [ ]:
corrected = (
        denoised
        - background
)

In [ ]:
corrected = np.clip(
    corrected,
    0,
    None
)

In [ ]:
corrected_max = corrected.max()

if corrected_max > 0:

    corrected = (
            corrected
            / corrected_max
    )

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6)
)

axes[0].imshow(
    normalized[z],
    cmap="gray"
)

axes[0].set_title("Raw normalized")

axes[1].imshow(
    denoised[z],
    cmap="gray"
)

axes[1].set_title("Denoised")

axes[2].imshow(
    corrected[z],
    cmap="gray"
)

axes[2].set_title(
    "Background corrected"
)

for ax in axes:
    ax.axis("off")

plt.show()

In [ ]:
plt.figure(figsize=(8, 8))

plt.imshow(
    corrected[z],
    cmap="magma"
)

plt.colorbar(
    label="Normalized intensity"
)

plt.title(
    f"Background-corrected — z={z}"
)

plt.axis("off")

plt.show()

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

axes[0].hist(
    normalized.ravel(),
    bins=100
)

axes[0].set_title(
    "Normalized intensity"
)

axes[1].hist(
    denoised.ravel(),
    bins=100
)

axes[1].set_title(
    "Denoised intensity"
)

axes[2].hist(
    corrected.ravel(),
    bins=100
)

axes[2].set_title(
    "Background-corrected intensity"
)

plt.show()

In [ ]:
from pathlib import Path
import numpy as np

OUTPUT_DIR = Path("../data/sample/processed")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = OUTPUT_DIR / "44b6_0113de3b_t00_z20_corrected.npy"

np.save(
    output_path,
    corrected
)

print("Saved:", output_path)
print("Shape:", corrected.shape)
print("Dtype:", corrected.dtype)